# Laws of RankMe Evolution

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy

In this notebook, we investigate whether different languages share the same geometric phases in their RankMe training curves, regardless of magnitude. We approach in two ways: by analyzing normalized curves, and by fitting shared parametric laws across all languages. We focus on the RankMe of last-token representations at the last layer.

In [ ]:
import pandas as pd
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.insert(0, os.path.join(REPO_ROOT, "code", "visualization"))
sys.path.insert(0, os.path.join(REPO_ROOT, "code", "geometry_analysis"))

from visualize_shaded_pt_curves import show_training_curves
from fit_rankme import fit_rankme_from_df, plot_fitted_laws, FuxiSharedACModel, FuxiPerLangACModel, ApertusSharedACModel, ApertusPerLangACModel, ApertusDualTailACModel, ApertusDualLamACModel

In [ ]:
def display_parameter_tables(params_df):
    skip = {"language", "r2", "sse", "layer", "aggregation"}
    numeric_cols = [c for c in params_df.columns if c not in skip]

    shared_cols = [c for c in numeric_cols if params_df[c].nunique() == 1]
    per_lang_cols = [c for c in numeric_cols if params_df[c].nunique() > 1]

    print("Shared parameters:")
    display(params_df[shared_cols].iloc[[0]].reset_index(drop=True).round(4))

    print("Per-language parameters:")
    display(params_df[["language"] + per_lang_cols + ["r2"]].reset_index(drop=True).round(4))

## FuxiTranyu 8B

We first visualize the RankMe curves of last-token representations at the last layer for FuxiTranyu 8B.

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, "code", "results", "fuxi_fine_wiki.csv")

fuxi_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(fuxi_df):,} rows from {CSV_PATH}")
fuxi_df.head()

In [ ]:
checkpoints = sorted(fuxi_df["checkpoint"].unique(), key=lambda x: int(x.replace("B", "")) if x[0].isdigit() else float("inf"))
languages = sorted(fuxi_df["dataset"].unique())
layers = sorted(fuxi_df["layer"].unique(), key=lambda x: int(x.split("_")[1]))
aggregations = sorted(fuxi_df["aggregation"].unique())
metric_cols = [c for c in fuxi_df.columns if c not in ("checkpoint", "dataset", "layer", "aggregation")]

print(f"Total rows: {len(fuxi_df):,}")
print(f"Checkpoints ({len(checkpoints)}): {checkpoints}")
print(f"Languages ({len(languages)}): {languages}")
print(f"Layers ({len(layers)}): {[layer.split('_')[1] for layer in layers]}")
print(f"Aggregations ({len(aggregations)}): {aggregations}")
print(f"Metrics ({len(metric_cols)}): {metric_cols}")

In [ ]:
show_training_curves(fuxi_df, metrics=["rankme"], layers=["layer_29"], aggregations=["last"], main_value=600, smoothing=1, path="../plots/report/fuxi", img_format="svg")

### Normalized curves

We normalize curves by applying min-max scaling independently within each phase (per-segment normalization), mapping each language's curve to [0, 1] per phase to remove magnitude differences while preserving shape.

In [ ]:
show_training_curves(fuxi_df, metrics=["rankme"], layers=["layer_29"], aggregations=["last"], smoothing=1, main_value=600, normalize="per-segment", path="../plots/report/fuxi", img_format="svg")

### Shared power-law + saturation model

The normalized curves show that all languages follow the same shape, differing only in magnitude and offset. This motivates a model $\hat{R}(t)$ of RankMe at time $t$ of the form

$$\hat{R}(t) = \alpha + \beta \, f(t)$$

where $\alpha$ is a per-language offset, $\beta$ is a per-language magnitude, and $f$ is a shared shape function. 

The curves show an initial power-law decrease followed by a bounded recovery, so we define $f$ as a two-phase piecewise function:

$$f(t) = \begin{cases} t^{-\gamma} & t \le t_1 \\ P_1 + C\!\left(1 - e^{-(t-t_1)/\lambda}\right) & t > t_1 \end{cases}$$

where $P_1 = t_1^{-\gamma}$ ensures continuity, and $\gamma$, $C$ and $\lambda$ are shared parameters. In particular:
- $\gamma$ is the power-law decay exponent, 
- $C$ is the amplitude of the phase-2 recovery, 
- $\lambda$ is the timescale of phase-2 recovery.

In [ ]:
fuxi_shared_model = FuxiSharedACModel()
fuxi_shared_params_df = fit_rankme_from_df(fuxi_df, layer="layer_29", aggregation="last", model=fuxi_shared_model, changepoints=[241])
display_parameter_tables(fuxi_shared_params_df)

In [ ]:
plot_fitted_laws(fuxi_shared_params_df, fuxi_df, model=fuxi_shared_model)

### Per-language amplitude model

The shared model achieves R² below 0.70 for Chinese, suggesting the amplitude varies per language even if the shape does not. We therefore make $A$ and $C$ per-language linear parameters, keeping only $\gamma$ and $\lambda$ shared.

The power law is clamped at its plateau value after the changepoint, so the model is piecewise and continuous at $t_1$:

$$\hat{R}(t) = \begin{cases} \alpha + A\,t^{-\gamma} & t \le t_1 \\ \alpha + A\,P_1 + C\!\left(1 - e^{-(t-t_1)/\lambda}\right) & t > t_1 \end{cases}$$

where $P_1 = t_1^{-\gamma}$.

In [ ]:
fuxi_perlang_model = FuxiPerLangACModel()
fuxi_perlang_params_df = fit_rankme_from_df(fuxi_df, layer="layer_29", aggregation="last", model=fuxi_perlang_model, changepoints=[241])
display_parameter_tables(fuxi_perlang_params_df)

In [ ]:
plot_fitted_laws(fuxi_perlang_params_df, fuxi_df, model=fuxi_perlang_model)

Making $A$ and $C$ per-language brings all 14 languages above R² = 0.90 (minimum: Swahili at 0.91), confirming that $\gamma$ and $\lambda$ are shared across languages.

## Apertus 8B

We first visualize the RankMe curves of last-token representations at the last layer for Apertus 8B.

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, "code", "results", "apertus_fine_wiki.csv")

apertus_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(apertus_df):,} rows from {CSV_PATH}")
apertus_df.head()

In [ ]:
checkpoints = sorted(apertus_df["checkpoint"].unique(), key=lambda x: int(x.replace("B", "")) if x[0].isdigit() else float("inf"))
languages = sorted(apertus_df["dataset"].unique())
layers = sorted(apertus_df["layer"].unique(), key=lambda x: int(x.split("_")[1]))
aggregations = sorted(apertus_df["aggregation"].unique())
metric_cols = [c for c in apertus_df.columns if c not in ("checkpoint", "dataset", "layer", "aggregation")]

print(f"Total rows: {len(apertus_df):,}")
print(f"Checkpoints ({len(checkpoints)}): {checkpoints}")
print(f"Languages ({len(languages)}): {languages}")
print(f"Layers ({len(layers)}): {[layer.split('_')[1] for layer in layers]}")
print(f"Aggregations ({len(aggregations)}): {aggregations}")
print(f"Metrics ({len(metric_cols)}): {metric_cols}")

In [ ]:
show_training_curves(apertus_df, metrics=["rankme"], layers=["layer_31"], aggregations=["last"], smoothing=0.2, path="../plots/report/apertus", img_format="svg")

We now apply the same per-phase min-max scaling already used for FuxiTranyu.

In [ ]:
show_training_curves(apertus_df, metrics=["rankme"], layers=["layer_31"], aggregations=["last"], smoothing=0.20, normalize="per-segment", path="../plots/report/apertus", img_format="svg")

### Three-phase shared model

The normalized Apertus curves show language sharing the same shape in the first two phases, but largely differing in the last one. Despite this, we try to use the same $\hat{R}(t) = \alpha + \beta\,f(t)$ structure already applied with FuxiTranyu, with $\alpha, \beta$ per-language and $f$ shared.

The curves exhibit two inflections: a plateau beginning around $t_1 \approx 630\,\text{B}$ and a second transition around $t_2 \approx 7.6\,\text{T}$. Therefore, we extend $f$ to three phases:

$$f(t) = \begin{cases} t^{-\gamma} & t \le t_1 \\ P_1 + C_2\!\left(1 - e^{-(t-t_1)/\lambda_2}\right) & t_1 < t \le t_2 \\ P_2 + C_3\!\left(1 - e^{-(t-t_2)/\lambda_3}\right) & t > t_2 \end{cases}$$

where $P_1 = t_1^{-\gamma}$ and $P_2 = P_1 + C_2\!\left(1 - e^{-(t_2-t_1)/\lambda_2}\right)$ ensure continuity, and $\gamma, C_2, \lambda_2, C_3, \lambda_3$ are shared across all languages. In particular:
- $\gamma$ is the power-law decay exponent,
- $C_2$ is the amplitude of the phase-2 recovery,
- $\lambda_2$ is the timescale of phase-2 recovery,
- $C_3$ is the amplitude of the phase-3 change,
- $\lambda_3$ is the timescale of phase-3 change.

In [ ]:
apertus_shared_model = ApertusSharedACModel()
apertus_shared_params_df = fit_rankme_from_df(apertus_df, "layer_31", "last", model=apertus_shared_model, changepoints=[630, 7652], t_scale=1000)
display_parameter_tables(apertus_shared_params_df)

In [ ]:
plot_fitted_laws(apertus_shared_params_df, apertus_df, model=apertus_shared_model)

### Three-phase per-language amplitude model

The shared model fails for Arabic, Swahili, Hindi and Vietnamese (R²<0.2). These four languages display a final expansion in the third phase, while the remaining languages compress. We attempt to fix the poor fit by making $A$, $C_2$, $C_3$ per-language linear parameters, keeping only $\gamma, \lambda_2, \lambda_3$ shared.

Each phase's basis is clamped at its plateau value at the next changepoint, giving a piecewise-continuous model:

$$\hat{R}(t) = \begin{cases} \alpha + A\,t^{-\gamma} & t \le t_1 \\ \alpha + A\,P_1 + C_2\!\left(1 - e^{-(t-t_1)/\lambda_2}\right) & t_1 < t \le t_2 \\ \alpha + A\,P_1 + C_2\,Q_2 + C_3\!\left(1 - e^{-(t-t_2)/\lambda_3}\right) & t > t_2 \end{cases}$$

where $P_1 = t_1^{-\gamma}$ and $Q_2 = 1 - e^{-(t_2-t_1)/\lambda_2}$.

In [ ]:
apertus_model_per_lang = ApertusPerLangACModel()
apertus_params_per_language_df = fit_rankme_from_df(apertus_df, "layer_31", "last", model=apertus_model_per_lang, changepoints=[630, 7652], t_scale=1000)
display_parameter_tables(apertus_params_per_language_df)

In [ ]:
plot_fitted_laws(apertus_params_per_language_df, apertus_df, model=apertus_model_per_lang)

### Dual-tail model (separate growth and decay)

The per-language model achieves R² above 0.90 for most languages, but still does not fit well the expanding languages. We replace the single phase-3 term with two mutually exclusive basis functions ($C_{3,d}, C_{3,g} \ge 0$, exactly one active per language) to model the expanding and compressing regimes separately. Both are instances of the unified basis

$$b(t, \lambda) = \left(e^{(t-t_2)/\lambda} - 1\right)\mathbf{1}[t > t_2]$$

which saturates in $(-1, 0]$ when $\lambda < 0$ and grows exponentially in $[0, +\infty)$ when $\lambda > 0$. Our model becomes:

$$\hat{R}(t) = \begin{cases} \alpha + A\,t^{-\gamma} & t \le t_1 \\ \alpha + A\,P_1 + C_2\!\left(1 - e^{-(t-t_1)/\lambda_2}\right) & t_1 < t \le t_2 \\ \alpha + A\,P_1 + C_2\,Q_2 + C_{3,d}\,b(t,-\lambda_3) & t > t_2 \;\text{(decay)} \\ \alpha + A\,P_1 + C_2\,Q_2 + C_{3,g}\,b(t,+\lambda_3) & t > t_2 \;\text{(growth)} \end{cases}$$

where $P_1 = t_1^{-\gamma}$ and $Q_2 = 1 - e^{-(t_2-t_1)/\lambda_2}$.

In [ ]:
apertus_model_per_lang_dual = ApertusDualTailACModel()
apertus_params_per_language_dual_df = fit_rankme_from_df(apertus_df, "layer_31", "last", model=apertus_model_per_lang_dual, changepoints=[630, 7652], t_scale=1000)
display_parameter_tables(apertus_params_per_language_dual_df)

In [ ]:
plot_fitted_laws(apertus_params_per_language_dual_df, apertus_df, model=apertus_model_per_lang_dual)

### Dual-$\lambda$ model (separate timescales per phase-3 branch)

The dual-tail model forces a single $\lambda_3$ for both decay and growth, but these dynamics have different timescales. We therefore fit separate $\lambda_{3,d}$ and $\lambda_{3,g}$, keeping $C_{3,d}, C_{3,g} \ge 0$ mutually exclusive per language:

$$\hat{R}(t) = \begin{cases} \alpha + A\,t^{-\gamma} & t \le t_1 \\ \alpha + A\,P_1 + C_2\!\left(1 - e^{-(t-t_1)/\lambda_2}\right) & t_1 < t \le t_2 \\ \alpha + A\,P_1 + C_2\,Q_2 + C_{3,d}\,b(t,-\lambda_{3,d}) & t > t_2 \;\text{(decay)} \\ \alpha + A\,P_1 + C_2\,Q_2 + C_{3,g}\,b(t,+\lambda_{3,g}) & t > t_2 \;\text{(growth)} \end{cases}$$

where $b(t,\lambda)$ is as defined above, $P_1 = t_1^{-\gamma}$, and $Q_2 = 1 - e^{-(t_2-t_1)/\lambda_2}$.

In [ ]:
apertus_model_dual_lam = ApertusDualLamACModel()
apertus_params_dual_lam_df = fit_rankme_from_df(apertus_df, "layer_31", "last", model=apertus_model_dual_lam, changepoints=[630, 7652], t_scale=1000)
display_parameter_tables(apertus_params_dual_lam_df)

In [ ]:
plot_fitted_laws(apertus_params_dual_lam_df, apertus_df, model=apertus_model_dual_lam)

Allowing separate timescales for the two branches brings all languages except Vietnamese (R² = 0.68) above R² = 0.86, with most above 0.90.

## Concluding observations

**FuxiTranyu.** All 14 languages follow nearly identical shapes, differing only in offset and magnitude. A fully shared model therefore achieves R² above 0.88 for 13 languages, with Chinese (R² = 0.69) as the main exception. Making $A$ and $C$ per-language while keeping $\gamma$ and $\lambda$ shared brings all languages above R² = 0.90.

**Apertus.** The third phase breaks cross-lingual shape agreement: Arabic, Hindi, Swahili and Vietnamese show a final RankMe expansion while the remaining languages compress. This causes shared models to fail on the expanding group (R² < 0.20 in the three-phase shared model). Progressively decoupling parameters (per-language $A$, $C_2$, $C_3$, then separate decay and growth branches with independent timescales) recovers a good fit for most languages, but Vietnamese remains problematic (R² ≈ 0.68).